# 06 · Phase 4: grounded explanations (RAG)

Same detector, same images, same evidence summaries and the same three Groq models as phase 3 (notebook 05).
The one change: every prompt now includes **evidence cards** retrieved from the forensic corpus (`corpus/`), and
the model must use and cite them. A third condition, **verified**, checks each grounded answer and asks once for
a revision when a check fails.

| Step | What happens |
|---|---|
| Retrieve | cards filtered by verdict and heatmap regions; ranked by region overlap, BM25 and embeddings (bge-small), fused by reciprocal rank fusion; the best card of 3 different artefact types, their scope cards and one limitation card |
| Generate | phase 3's prompt + the cards + the instruction to use and cite them (temperature 0, reasoning *low*) |
| Verify | deterministic checks against the cited cards, and an NLI model (DeBERTa-v3) that tests whether each sentence follows from the evidence summary and the cited cards |

**Notebook settings (right sidebar)**
- **Accelerator:** GPU T4 (detector, embeddings and NLI) · **Internet:** On
- **Inputs:** **01_extract_faces**, **01b_extract_celebdf**, **02_train** (the copy with EfficientNet-B0) and
  **05_explanations** (its saved output: the evidence table and the phase 3 answers)
- **Secrets:** `GROQ_API_KEY`

**Free-tier budget:** grounded prompts are about 1,000 tokens, so GPT-OSS requests are spaced 15 s apart (Qwen 30 s).
One run (grounded + verified) uses roughly 100–180K tokens per model, close to Groq's 200K daily limit, so run it
once per day; for a repeat run set `RUN_TAG = "run2"` the next day. Expect about 1.5 hours.

**Produces:** `/kaggle/working/outputs/grounded/` (every answer with the ids of the cards it was given and the raw
reply, scores per answer, summary tables, and a sentence sample for checking the NLI model by hand).

In [ ]:
# --- Project code -------------------------------------------------------------------------
# Always clones the latest code. It lives in /tmp, OUTSIDE /kaggle/working, so it is never saved into this
# notebook's output (otherwise later notebooks would pick up a stale copy). For a private repo add a
# GITHUB_TOKEN secret. Fallback: a zip of the project uploaded as a Kaggle *dataset*.
import os, sys, glob, shutil, subprocess

PROJECT = "/tmp/dfdc-project"
REPO, BRANCH = "https://github.com/Amay-M-Nair/DFDC.git", "dev"

if not os.path.exists(PROJECT):
    token = ""
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT], capture_output=True)
    if r.returncode == 0:
        subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", REPO])  # don't keep the token
    else:
        found = [p for d in range(5) for p in glob.glob("/kaggle/input/" + "*/" * d + "src/config.py")]
        if not found:
            raise RuntimeError("git clone failed: add a GITHUB_TOKEN secret or attach the project zip as a dataset")
        print("WARNING: git clone failed; using the attached copy at", os.path.dirname(os.path.dirname(found[0])))
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), PROJECT,
                        ignore=shutil.ignore_patterns("data", "outputs", "__pycache__"))

shutil.rmtree("/kaggle/working/project", ignore_errors=True)  # remove stale copies saved by older notebook versions
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip install -q grad-cam openai
!git -C {PROJECT} log -1 --format="code version: %h  %s  (%cd)" --date=short 2>/dev/null || echo "code version: attached copy (no git)"

In [ ]:
%matplotlib inline
import ast, json, os, re, time
from collections import Counter
from dataclasses import fields
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown

from src.kaggle_utils import setup, find, find_datasets, checkpoint_roots, setup_llm
setup(data=False)

from src.explain import retriever as retrieval
assert hasattr(retrieval.Retriever, "retrieve"), "Outdated code: restart the session (Run → Restart) and run all again."
from src.classifier.compare import find_checkpoints
from src.config import IMG_SIZE
from src.eval.verify import EvidenceVerifier, NLIChecker
from src.explain.corpus import METHOD_FAMILY, load_corpus
from src.explain.llm import GROUNDED_SUFFIX, ForensicExplainer, LLMExplanation
from src.explain.retriever import DenseEncoder, Retriever
from src.xai.gradcam import Explainer, draw_bbox, draw_regions

DATASETS = {k: v for k, v in find_datasets().items() if k in ("FF++", "CelebDF")}
CKPTS = find_checkpoints(checkpoint_roots())
P3 = find("evidence.csv")
assert P3 is not None, "Phase 3 output not found: add the saved output of 05_explanations as an input."
P3 = P3.parent
OUT = Path("/kaggle/working/outputs/grounded")
OUT.mkdir(parents=True, exist_ok=True)
print("datasets:", DATASETS)
print("detectors available:", list(CKPTS))
print("phase 3 output:", P3, sorted(p.name for p in P3.glob("phase3_*.jsonl")))
cards = load_corpus()
print(f"corpus: {len(cards)} cards; {sum(not c.checked for c in cards)} not yet checked against their source")

## Settings

In [ ]:
DETECTOR = "efficientnet_b0"                  # chosen in notebook 04b
LLMS = [("groq", "openai/gpt-oss-120b"),      # same three models and settings as phase 3
        ("groq", "openai/gpt-oss-20b"),
        ("groq", "qwen/qwen3.8-27b")]
MIN_INTERVAL = {"groq": 15}                   # seconds between requests; grounded prompts are ~1K tokens (phase 3: 12 s)
MODEL_LIMITS = {"qwen/qwen3.8-27b": {"min_interval": 30, "max_tokens": 900}}  # Groq: 1K output tokens/min for Qwen
os.environ["LLM_REASONING_EFFORT"] = "low"    # same reasoning effort for every model
RANK_BY = "concentration"                     # as in phase 3
RANKERS = ("region", "bm25", "dense")         # retriever for the run (section 2 compares the alternatives)
VERIFY = True                                 # also run the verified condition (one revision when a check fails)
REVISE_ON = ("contradiction",)                # NLI labels that trigger a revision; add "neutral" for a stricter loop
RUN_TAG = "run1"                              # change for a repeat run, e.g. "run2" on another day
assert DETECTOR in CKPTS, f"{DETECTOR} not found: attach the 02_train copy that trained it"

## 1 · Same images and evidence as phase 3
The LLM gets exactly phase 3's evidence summaries, read from notebook 05's evidence table. The detector runs
again only to rebuild the heatmaps for the pointing and IoU checks; the cell checks that the verdict and the
top-3 regions match phase 3.

In [ ]:
class Evidence:
    # Recomputed heatmap and regions, but phase 3's exact evidence summary (what the LLM sees).
    def __init__(self, exp, summary):
        self.exp, self.summary = exp, summary

    def __getattr__(self, name):
        return getattr(self.exp, name)

    def describe(self, k=3):
        return self.summary


ev = pd.read_csv(P3 / "evidence.csv")
explainer = Explainer(CKPTS[DETECTOR], rank_by=RANK_BY)
items, mismatched, changed_text = [], [], 0
for r in ev.itertuples():
    path = Path(DATASETS[r.dataset], "test", r.label, r.file)
    exp = explainer(Image.open(path))
    if exp.label != r.pred or [n for n, _ in exp.top_regions[:3]] != ast.literal_eval(r.top_regions):
        mismatched.append(r.file)
    changed_text += exp.describe() != r.summary
    items.append({"set": r.set, "dataset": r.dataset, "file": r.file, "path": str(path),
                  "manipulation": r.manipulation, "label": r.label, "exp": Evidence(exp, r.summary)})
del explainer
print(ev.groupby(["set", "dataset"]).size().to_string())
print(f"{len(items)} images; verdict or top-3 regions differ from phase 3 on {len(mismatched)}; "
      f"summary text differs on {changed_text} (the LLM still gets phase 3's text)")
if mismatched:
    print("WARNING, check these:", mismatched[:10])

## 2 · Retrieval
What the retriever offers on the pilot images, for several retriever variants (all of them filter by verdict).
Checks on FAKE verdicts:
- **artefact types offered**: how many different artefact types the LLM can choose from (target 3)
- **top region covered**: an offered artefact card lists the heatmap's top region
- **true method covered**: an offered artefact card is documented for the image's real manipulation family. The
  retriever never sees the method; this shows whether a method-consistent answer is available at all
- **GAN card offered**: a GAN-fingerprint card is offered (none of these datasets is fully GAN-generated)
- **distinct card sets**: how much retrieval varies between images

In [ ]:
encoder = DenseEncoder()
VARIANTS = {"region only": Retriever(("region",), encoder=encoder),
            "BM25 only": Retriever(("bm25",), encoder=encoder),
            "embeddings only": Retriever(("dense",), encoder=encoder),
            "region + BM25": Retriever(("region", "bm25"), encoder=encoder),
            "region + BM25 + embeddings": Retriever(("region", "bm25", "dense"), encoder=encoder),
            "same, plain top-3 (no region filter, no diversity)": Retriever(RANKERS, filter_regions=False,
                                                                            diversify=False, encoder=encoder)}


def retrieval_stats(ret):
    rows, sets = [], set()
    for it in items:
        exp = it["exp"]
        if it["set"] != "pilot" or exp.label != "fake":
            continue
        docs = ret.retrieve(exp)
        sets.add(tuple(d.id for d in docs))
        art = [d for d in docs if d.kind == "artifact"]
        family = METHOD_FAMILY.get(it["manipulation"])
        rows.append({"artefact types offered": len({d.artifact_type for d in art}),
                     "top region covered": any(exp.top_regions[0][0] in d.regions for d in art),
                     "true method covered": any(d.applies_to(family) for d in art) if family else np.nan,
                     "GAN card offered": any(d.artifact_type == "spectral_gan_fingerprint" for d in art)})
    return {**pd.DataFrame(rows).astype(float).mean().round(2).to_dict(), "distinct card sets": len(sets)}


display(pd.DataFrame({name: retrieval_stats(r) for name, r in VARIANTS.items()}).T)
RETRIEVER = Retriever(RANKERS, encoder=encoder)
offered = Counter(d.artifact_type or d.kind for it in items if it["set"] == "pilot" for d in RETRIEVER.retrieve(it["exp"]))
print(f"run retriever: {RETRIEVER.describe()}; cards offered on the pilot by type:", dict(offered.most_common()))

## 3 · Exactly what the LLM sees

In [ ]:
assert setup_llm(*LLMS[0]), "Add the GROQ_API_KEY secret (Add-ons → Secrets)"
system, user, docs = ForensicExplainer(retriever=RETRIEVER).build_prompt(items[0]["exp"], grounded=True)
display(Markdown("**Added to phase 3's system prompt:**"))
print(GROUNDED_SUFFIX.strip())
display(Markdown(f"**User message for the first example** ({len((system + user).split())} words in total):"))
print(user)

## 4 · Run each LLM: grounded, then verified
**Grounded:** one request per image with the cards. **Verified:** each grounded answer is checked; if a check
fails (invalid output, region outside the top-3, artefact contradicting the verdict, citations missing or not
provided, artefact not supported by a cited card, or a sentence the NLI model finds contradicted), the model gets
one revision request listing the problems. Answers that pass are kept as they are.

In [ ]:
verifier = EvidenceVerifier(NLIChecker(), revise_on=REVISE_ON)
exps = [it["exp"] for it in items]
results, run_log = {}, []
for provider, model_id in LLMS:
    print(f"\n===== {provider}: {model_id} =====")
    t0 = time.time()
    limits = MODEL_LIMITS.get(model_id, {})
    os.environ["LLM_MIN_INTERVAL"] = str(limits.get("min_interval", MIN_INTERVAL.get(provider, 0)))
    if "max_tokens" in limits:
        os.environ["LLM_MAX_TOKENS"] = str(limits["max_tokens"])
    else:
        os.environ.pop("LLM_MAX_TOKENS", None)    # the provider's default (1024 on Groq)
    try:
        if not setup_llm(provider, model_id):
            raise RuntimeError("add the GROQ_API_KEY secret")
        llm = ForensicExplainer(retriever=RETRIEVER)
        conditions = {"grounded": llm.explain_batch(exps, grounded=True)}
        if VERIFY:
            conditions["verified"] = llm.revise_batch(exps, conditions["grounded"], verifier)
        name = model_id.replace("/", "__")
        for cond, outs in conditions.items():
            results[(cond, model_id)] = outs
            rows = [{"llm": model_id, "provider": provider, "run": RUN_TAG, "retriever": RETRIEVER.describe(),
                     **{k: v for k, v in it.items() if k not in ("exp", "path")}, **o.to_dict()}
                    for it, o in zip(items, outs)]
            with open(OUT / f"phase4_{cond}_{name}_{RUN_TAG}.jsonl", "w") as f:
                f.writelines(json.dumps(r) + "\n" for r in rows)
        g = conditions["grounded"]
        log = {"llm": model_id, "status": "done", "minutes": round((time.time() - t0) / 60, 1),
               "grounded_valid": f"{np.mean([o.schema_ok for o in g]):.0%}",
               "request_errors": sum(bool(o.error) and not o.raw for o in g)}
        if VERIFY:
            v = conditions["verified"]
            log.update(revised=sum(o.revised for o in v), verified_valid=f"{np.mean([o.schema_ok for o in v]):.0%}")
        run_log.append(log)
        del llm
    except Exception as e:
        print(f"SKIPPED {model_id}: {type(e).__name__}: {e}")
        run_log.append({"llm": model_id, "status": f"failed: {type(e).__name__}: {e}"[:120],
                        "minutes": round((time.time() - t0) / 60, 1)})
display(pd.DataFrame(run_log))
first_error = next((o.error for outs in results.values() for o in outs if o.error and not o.raw), "")
if first_error:
    print("first request error:", first_error[:300])
assert results, "No LLM produced explanations; see the errors above."

## 5 · Ungrounded vs. grounded vs. verified (pilot images)
Phase 3's answers (from notebook 05) are scored with the same checks. Each image counts once per model and
condition. Rates are shares of answers; checks that do not apply are left out of a rate.

| Check | Meaning |
|---|---|
| valid | JSON with labels from the allowed lists |
| region_in_top3 · pointing_hit · region_iou | the named region against the heatmap (top-3, hottest pixel, overlap with the top 20%) |
| artifact_plausible | the artefact occurs in one of the top-3 regions (corpus taxonomy) |
| verdict_consistent | an artefact for FAKE, `none` for REAL |
| citation_valid · cites_cards | citations were provided to the model · it cites at least one (grounded only) |
| artifact_in_cited_card · region_in_cited_card | a cited artefact card supports the artefact · and lists the region (FAKE, grounded only) |
| method_consistent | the artefact is documented for the image's true manipulation family (FAKE answers on fake images) |
| gan_claim | the answer claims a GAN fingerprint (lower is better: no image here is fully GAN-generated) |
| supported_share · contradicted_share | sentences the NLI model finds entailed / contradicted by the evidence summary + cited cards |

NLI measures attribution, not truth: a correct sentence from the model's own knowledge counts as unsupported,
and hedged interpretations are often judged neutral. Check it by hand on the exported sample before relying on it.

In [ ]:
names = {f.name for f in fields(LLMExplanation)}
for provider, model_id in LLMS:
    path = P3 / f"phase3_{model_id.replace('/', '__')}.jsonl"
    if path.exists():
        by_key = {(r["set"], r["file"]): r for r in map(json.loads, open(path))}
        results[("ungrounded", model_id)] = [LLMExplanation(**{k: v for k, v in by_key[(it["set"], it["file"])].items()
                                                               if k in names}) for it in items]

rows, sentence_rows = [], []
for (cond, model_id), outs in results.items():
    pilot = [(it, o) for it, o in zip(items, outs) if it["set"] == "pilot"]
    support = verifier.support_batch([o for _, o in pilot], [it["exp"] for it, _ in pilot])
    for (it, o), s in zip(pilot, support):
        rows.append({"condition": cond, "llm": model_id.split("/")[-1], "file": it["file"],
                     "manipulation": it["manipulation"], "verdict": it["exp"].label,
                     "verdict_correct": it["exp"].label == it["label"], "confidence": it["exp"].confidence,
                     "artifact": o.artifact_type if o.schema_ok else None, "revised": o.revised,
                     **verifier.checks(o, it["exp"], it["manipulation"], s)})
        sentence_rows += [{"condition": cond, "llm": model_id.split("/")[-1], "file": it["file"], "sentence": sent,
                           "nli_label": label, "premise": verifier.premise(o, it["exp"])}
                          for sent, label in zip(s["sentences"], s["labels"])]
scores = pd.DataFrame(rows)
METRICS = [m for m in ["valid", "region_in_top3", "pointing_hit", "region_iou", "artifact_plausible",
                       "verdict_consistent", "citation_valid", "cites_cards", "artifact_in_cited_card",
                       "region_in_cited_card", "method_consistent", "gan_claim", "supported_share",
                       "contradicted_share"] if m in scores]
scores[METRICS] = scores[METRICS].astype(float)
ORDER = [c for c in ("ungrounded", "grounded", "verified") if c in set(scores.condition)]
table = scores.groupby(["llm", "condition"])[METRICS].mean().round(3)
table = table.reindex(pd.MultiIndex.from_product([sorted(scores.llm.unique()), ORDER])).dropna(how="all")
display(table)
overall = scores.groupby("condition")[METRICS].mean().reindex(ORDER).round(3)
display(Markdown("**All three models together:**"))
display(overall)


def fleiss_kappa(labels: pd.DataFrame) -> float:
    cats = sorted(set(labels.values.ravel()))
    counts = np.array([[list(row).count(c) for c in cats] for row in labels.values])
    n = counts.sum(1)[0]
    p_item = ((counts ** 2).sum(1) - n) / (n * (n - 1))
    p_cat = counts.sum(0) / counts.sum()
    expected = (p_cat ** 2).sum()
    return float((p_item.mean() - expected) / (1 - expected)) if expected < 1 else 1.0


agreement = []
for cond in ORDER:
    w = scores[(scores.condition == cond) & (scores.verdict == "fake")].pivot_table(
        index="file", columns="llm", values="artifact", aggfunc="first").dropna()
    if w.shape[1] >= 2 and len(w):
        agreement.append({"condition": cond, "fake images": len(w),
                          "Fleiss kappa (artefact type)": round(fleiss_kappa(w), 3),
                          "all models agree": round(float((w.nunique(axis=1) == 1).mean()), 3)})
display(Markdown("**Do the models agree on the artefact type?** (FAKE verdicts answered validly by every model)"))
display(pd.DataFrame(agreement))
display(Markdown("**Correct vs. wrong detector verdicts** (explanations of a wrong verdict are still scored)"))
display(scores.groupby(["condition", "verdict_correct"])[[m for m in ["supported_share", "region_in_top3",
        "method_consistent"] if m in METRICS]].mean().reindex(ORDER, level=0).round(3))

if "verified" in ORDER:
    v = scores[scores.condition == "verified"]
    print(f"revised: {int(v.revised.sum())} of {len(v)} verified answers")
    reasons = Counter(re.sub(r'"[^"]*"|\(.*?\)', "…", p)[:70] for (cond, _), outs in results.items()
                      if cond == "verified" for o in outs for p in o.problems)
    print("most common problems:", reasons.most_common(6))

scores.to_csv(OUT / f"phase4_scores_{RUN_TAG}.csv", index=False)
table.to_csv(OUT / f"phase4_summary_{RUN_TAG}.csv")
pd.DataFrame(agreement).to_csv(OUT / f"phase4_agreement_{RUN_TAG}.csv", index=False)
sample = pd.DataFrame(sentence_rows).sample(n=min(60, len(sentence_rows)), random_state=0)
sample.assign(your_label="").to_csv(OUT / f"nli_check_sample_{RUN_TAG}.csv", index=False)

plot = [m for m in ["region_in_top3", "pointing_hit", "artifact_plausible", "method_consistent", "supported_share",
                    "gan_claim", "contradicted_share"] if m in overall]
ax = overall[plot].T.plot.bar(figsize=(11, 3.6), rot=0, width=0.8, ylim=(0, 1.05))
ax.set_ylabel("share of answers")
ax.set_title("Phase 4: ungrounded vs. grounded vs. verified (gan_claim and contradicted_share: lower is better)",
             loc="left", fontsize=10)
ax.grid(axis="y", color="#d9d9d4", linewidth=0.6); ax.set_axisbelow(True)
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), frameon=False)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig(OUT / f"phase4_{RUN_TAG}.png", dpi=150); plt.show()
print("saved:", sorted(p.name for p in OUT.iterdir()))

## 6 · Worked examples
For each example: the evidence, the cards the grounded prompt offered, and every model's answer in each condition
with the problems the verifier finds (✅ = passes every check).

In [ ]:
def verdict_cell(o, exp):
    if not o.schema_ok:
        return "⚠️ invalid output" + (f" ({o.error[:60]})" if o.error else "")
    found = verifier.problems(o, exp)
    return "✅" if not found else "❌ " + "; ".join(p[:70] for p in found[:3])


for i, it in enumerate(items):
    if it["set"] != "example":
        continue
    exp = it["exp"]
    rgb = np.asarray(Image.open(it["path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE)))
    fig, axes = plt.subplots(1, 3, figsize=(7.5, 2.6))
    axes[0].imshow(rgb); axes[1].imshow(draw_bbox(exp.overlay, exp.bbox)); axes[2].imshow(draw_regions(rgb, exp.regions))
    for ax, t in zip(axes, ("face", "Grad-CAM++ evidence", "facial regions")):
        ax.set_title(t, fontsize=8); ax.axis("off")
    truth = "real" if it["manipulation"] == "original" else f"fake ({it['manipulation']})"
    plt.suptitle(f"{it['dataset']} · truth: {truth} · detector: {exp.label} ({exp.prob_fake:.3f}, {exp.confidence})", fontsize=9)
    plt.tight_layout(); plt.show()
    offered = ", ".join(f"`{d.id}`" for d in RETRIEVER.retrieve(exp))
    rows = "".join(f"| `{m.split('/')[-1]}` | {cond}{' (revised)' if o.revised else ''} | {o.claimed_region or '—'} | "
                   f"`{o.artifact_type or '—'}` | {', '.join(o.citations) or '—'} | "
                   f"{(o.explanation or o.error or o.raw[:120]).replace('|', '/')} | {verdict_cell(o, exp)} |\n"
                   for _, m in LLMS for cond in ORDER if (cond, m) in results for o in [results[(cond, m)][i]])
    display(Markdown(f"*Evidence summary:* {exp.describe()}\n\n*Cards offered:* {offered}\n\n"
                     f"| LLM | condition | region | artifact | cites | explanation | verifier |\n"
                     f"|---|---|---|---|---|---|---|\n{rows}"))

## Before reporting these numbers
1. **Check the cards.** Every card in `corpus/` says `checked: no` until its claim has been compared with the
   cited paper. Change it to `checked: yes` once done (the count is printed at the top of this notebook).
2. **Check the NLI model.** Label the sentences in `nli_check_sample_<run>.csv` yourself (`your_label`: entailment,
   neutral or contradiction) and compare with `nli_label` before quoting the supported/contradicted shares.
3. **Repeat the run** on another day (`RUN_TAG = "run2"`): phase 3 showed run-to-run differences of up to 6 points.